Cell 1 — Imports

In [1]:
import json
import time
from pathlib import Path

import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

Cell 2 — Config

In [2]:
BASE_URL = "https://www.fintelligence.ir"

AUTHOR_ID = 4
AUTHOR_NAME = "دکتر کمیل رودی"

LINKS_FILE = Path("data/raw/webpages/komeil_roudi_links.txt")
OUTPUT_DIR = Path("data/raw/webpages/komeil_roudi")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

session = requests.Session()

session.headers.update(
    {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/149.0.0.0 Safari/537.36"
        )
    }
)

Cell 3 — Get Author Links

In [3]:
def get_author_links():

    page = 1
    links = []

    while True:

        print(f"Reading page {page}")

        url = (
            f"{BASE_URL}/Author/GetAuthorBlogByPage/"
            f"{page}?auid={AUTHOR_ID}"
        )

        response = session.get(url, timeout=30)

        if response.status_code != 200:
            break

        soup = BeautifulSoup(response.text, "html.parser")

        page_links = []

        for a in soup.find_all("a", href=True):

            href = a["href"]

            if "/Blog/Detail/" in href:

                if href.startswith("/"):
                    href = BASE_URL + href

                page_links.append(href)

        page_links = sorted(set(page_links))

        if not page_links:
            break

        print(f"  {len(page_links)} links")

        links.extend(page_links)

        page += 1

    links = sorted(set(links))

    LINKS_FILE.parent.mkdir(parents=True, exist_ok=True)

    with open(LINKS_FILE, "w", encoding="utf-8") as f:
        for link in links:
            f.write(link + "\n")

    print("\n" + "=" * 40)
    print(f"Finished")
    print(f"Total articles : {len(links)}")
    print(f"Saved to       : {LINKS_FILE}")

    return links

Cell 4 (اجرای تابع)

In [4]:
links = get_author_links()

Reading page 1
  8 links
Reading page 2
  8 links
Reading page 3
  8 links
Reading page 4
  8 links
Reading page 5
  8 links
Reading page 6
  8 links
Reading page 7
  8 links
Reading page 8
  8 links
Reading page 9
  8 links
Reading page 10
  4 links
Reading page 11

Finished
Total articles : 76
Saved to       : data/raw/webpages/komeil_roudi_links.txt


Cell 5 — توابع کمکی

In [5]:
def clean_text(text):
    return " ".join(text.split())


def article_id(url):
    return url.split("/Blog/Detail/")[1].split("/")[0]

Cell 6 — دانلود و استخراج مقاله

In [6]:
def download_article(url):

    response = session.get(url, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    title = ""

    if soup.title:
        title = clean_text(soup.title.get_text())

    content_div = soup.select_one(".blog-post-body-container .mt-md")

    paragraphs = []

    if content_div:

        for p in content_div.find_all("p"):

            text = clean_text(
                p.get_text(" ", strip=True)
            )

            if len(text) > 20:
                paragraphs.append(text)

    body = "\n\n".join(dict.fromkeys(paragraphs))

    return {
        "url": url,
        "title": title,
        "content": body,
        "html": response.text,
    }

Cell 7 — تست روی یک مقاله

In [7]:
article = download_article(links[0])

print(article["title"])
print()
print(article["content"][:1000])

چطور شغل‌مان را توسعه دهیم؟




Cell 8 — دانلود و ذخیره همه مقالات

In [8]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

failed = []

for url in tqdm(links):

    try:

        article = download_article(url)

        aid = article_id(url)

        article_folder = OUTPUT_DIR / aid
        article_folder.mkdir(parents=True, exist_ok=True)

        # HTML
        with open(article_folder / "article.html", "w", encoding="utf-8") as f:
            f.write(article["html"])

        # JSON
        with open(article_folder / "article.json", "w", encoding="utf-8") as f:
            json.dump(
                {
                    "url": article["url"],
                    "title": article["title"],
                    "content": article["content"],
                },
                f,
                ensure_ascii=False,
                indent=2,
            )

        # Markdown
        with open(article_folder / "article.md", "w", encoding="utf-8") as f:
            f.write(f"# {article['title']}\n\n")
            f.write(article["content"])

        time.sleep(0.2)

    except Exception as e:

        print(f"Error: {url}")
        print(e)

        failed.append(url)

print("=" * 40)
print("Downloaded:", len(links) - len(failed))
print("Failed:", len(failed))

100%|██████████| 76/76 [00:36<00:00,  2.06it/s]

Downloaded: 76
Failed: 0


Cell 9 — بررسی صحت دانلود

In [9]:
folders = sorted(OUTPUT_DIR.iterdir())

print("Number of downloaded articles:", len(folders))

for folder in folders[:5]:
    print(folder.name)

Number of downloaded articles: 76
00D9A87F
0A050EE9
11140315
14509512
1699F2E4


Cell 10 — بررسی کیفیت فایل‌ها

In [10]:
empty_articles = []

for folder in folders:

    json_file = folder / "article.json"

    with open(json_file, encoding="utf-8") as f:
        article = json.load(f)

    if len(article["content"]) < 100:
        empty_articles.append(folder.name)

print("Empty articles:", len(empty_articles))

if empty_articles:
    print(empty_articles)

Empty articles: 17
['00D9A87F', '1699F2E4', '1B330C23', '24A50616', '2AA67199', '2E656F0E', '4A778FDE', '746AFDA4', '79FA69C8', '7C71884B', '847D7023', '913C0B94', '92B86E79', '9673DF51', 'A965DC32', 'BF2CFA69', 'EEB62FE1']




فقط یک پیشنهاد کوچک برای آینده (الان لازم نیست انجامش بدهی):

به جای اینکه `content` برای ویدئوها رشته خالی باشد، بهتر است بعداً در `download_article()` این را هم اضافه کنیم:

```python
is_video = (
    len(body) == 0 and
    content_div is not None and
    content_div.find("script") is not None
)
```

و داخل JSON ذخیره کنیم:

```json
{
  "url": "...",
  "title": "...",
  "content": "",
  "is_video": true
}
```

این باعث می‌شود در مراحل بعدی بتوانی خیلی راحت تمام ویدئوها را پیدا کنی و از آپارات Transcript یا زیرنویسشان را بگیری، بدون اینکه دوباره کل سایت را Crawl کنی.

به نظر من این بهترین نقطه پایان برای بخش **Website Ingestion** است و از اینجا می‌توانیم وارد مرحله **Preprocessing** شویم.
